# 🔬 Unsupervised Permutation Feature Importance: Pure Normal & Per-Class Analysis

This notebook measures the **Permutation Feature Importance** across the complete 20-feature set for our core unsupervised anomaly detection models:
- **Parametric Density / Distance**: `Mahalanobis Distance`, `Gaussian Mixture Model (GMM)`
- **Tree-Based Ensemble Isolation**: `Isolation Forest`
- **Deep Temporal Sequence Model**: `TCN Autoencoder`

### 20-Feature Set Analyzed:
1. **Raw Sensor (1)**: `course`
2. **Baseline Kinematics (10)**: `height`, `ground_speed`, `vertical_speed`, `acceleration`, `turn_rate`, `path_curvature`, `heading_speed_consistency`, `motion_smoothness`, `prediction_error`, `yaw_acceleration`
3. **Noise Textures (3)**: `prediction_error_autocorrelation`, `position_residual_std`, `speed_spectral_entropy`
4. **PE Macro-Trends (3)**: `pe_window_mean`, `pe_window_var`, `pe_window_skew`
5. **Cross-Correlations (3)**: `corr_speed_turn`, `corr_accel_turn`, `corr_vert_speed`

### Metrics Evaluated:
1. **Pure Model Importance (Normal Score Delta)**: Raw anomaly score / MSE reconstruction error increase on clean `Normal DJI` test telemetry when feature is permuted.
2. **Per-Class Detection Degradation ($|\Delta \text{TPR}|$)**: Drop in detection rate on each attack class (`Real ESP32`, `Sim Baseline`, `Sim Easy`, `Sim Medium`, `Sim Hard`, `Sim Geometry`).
3. **Class-Specific Grouped Horizontal Bar Charts**: Comparing how all 4 models respond to feature permutations on each specific dataset.

## 1. Setup Working Directory & Imports

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Idempotent repository root anchor (prevents cwd drift on repeated cell executions)
current = Path.cwd().resolve()
while current != current.parent and not (current / "implement").exists():
    current = current.parent
PROJECT_ROOT = current
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import seaborn as sns
import torch


from implement.utils.helper.features import ALL_INCLUSIVE_20_FEATURES
from presets.run_feature_importance import run_permutation_importance
from implement.utils.helper import get_output_dir

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Operating Directory: {os.getcwd()}")
print(f"Compute Device: {device} | PyTorch: {torch.__version__}")
print(f"20 Features Analyzed ({len(ALL_INCLUSIVE_20_FEATURES)}): {ALL_INCLUSIVE_20_FEATURES}")

## 2. Execute Complete Permutation Feature Importance Experiment

In [ ]:
df_master = run_permutation_importance(
    epochs=15,          # Set 15 for full training on GPU (or 1 for fast dry run)
    batch_size=64,
    window_len=20,
    k_thresh=3.0,
    fresh_cache=False,
    n_repeats=3,
    random_state=42
)

## 3. Master Feature Importance Table (Pure Normal + Per-Class Drops)

In [ ]:
csv_path = get_output_dir() / "feature_importance" / "master_feature_importance_per_class.csv"
if csv_path.exists():
    df_display = pd.read_csv(csv_path)
    print("\n=== MASTER PER-CLASS PERMUTATION FEATURE IMPORTANCE ===")
    display(df_display)
else:
    print("Run the experiment cell above to generate results.")

## 4. Visualizations: Class-Specific Grouped Horizontal Bar Charts
Displays side-by-side grouped horizontal bars for each model across all 20 features for every dataset class.

In [ ]:
from IPython.display import Image, display

out_dir = get_output_dir() / "feature_importance"
plot_files = [
    ("Pure Normal Anomaly Score Delta", "importance_pure_normal.png"),
    ("Overall Spoofed Detection Degradation", "importance_overall_spoofed.png"),
    ("Real ESP32 Feature Importance", "importance_real_esp32.png"),
    ("Sim Baseline Feature Importance", "importance_sim_baseline.png"),
    ("Sim Easy Feature Importance", "importance_sim_easy.png"),
    ("Sim Medium Feature Importance", "importance_sim_medium.png"),
    ("Sim Hard Feature Importance", "importance_sim_hard.png"),
    ("Sim Geometry Feature Importance", "importance_sim_geometry.png"),
]

for title, fname in plot_files:
    p = out_dir / fname
    if p.exists():
        print(f"\n{'======================================================================'}\n📊 {title}\n{'======================================================================'}")
        display(Image(filename=str(p)))